# 1. Imports

In [9]:
import pandas as pd
import re

import joblib
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, hamming_loss

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# 2. Loading datasets

In [2]:
df_train = pd.read_csv("/kaggle/input/datasets/saifsust/emonoba/Train.csv")
df_test = pd.read_csv("/kaggle/input/datasets/saifsust/emonoba/Test.csv")
df_val = pd.read_csv("/kaggle/input/datasets/saifsust/emonoba/Val.csv")

In [3]:
print(f"Train : {df_train.shape[0]}")
print(f"Test : {df_test.shape[0]}")
print(f"Val : {df_val.shape[0]}")

df_train.head(3)

Train : 18420
Test : 2272
Val : 2047


,ID,Data,Love,Joy,Surprise,Anger,Sadness,Fear,Topic,Domain,is_admin
0,5454,লকাল বাস ভালো এটা থেকে,0,0,0,0,1,0,Travel,Youtube,False
1,22549,কত অভিজানই তো চলে কিন্তু ওয়াসার পানির অভিজান ক...,0,0,0,0,1,0,Politics,Youtube,False
2,7033,বিয়ের মহল ছেড়ে তিনি বিস্রাম নিতে চলে যান (৬ ...,0,0,0,1,0,0,Personal,Facebook,False


# 3. Text preprocessing

In [4]:
def remove_punctuations(my_str):
    # define punctuation
    punctuations = '''````£|¢|Ñ+-*/=EROero৳০১২৩৪৫৬৭৮৯012–34567•89।!()-[]{};:'"“\’,<>./?@#$%^&*_~‘—॥”‰⚽️✌�￰৷￰'''
    
    no_punct = ""
    for char in my_str:
        if char not in punctuations:
            no_punct = no_punct + char

    # display the unpunctuated string
    return no_punct

def replace_strings(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           u"\u00C0-\u017F"          #latin
                           u"\u2000-\u206F"          #generalPunctuations
                               
                           "]+", flags=re.UNICODE)
    english_pattern=re.compile('[a-zA-Z0-9]+', flags=re.I)
    
    text=emoji_pattern.sub(r'', text)
    text=english_pattern.sub(r'', text)

    return text

def preprocessing(text):
    out=remove_punctuations(replace_strings(text))
    return out

In [25]:
df_train['Text'] = df_train.Data.apply(lambda x: preprocessing(str(x)))
df_train[['Text','Data']].head()

df_test['Text'] = df_test.Data.apply(lambda x: preprocessing(str(x)))
df_val['Text'] = df_val.Data.apply(lambda x: preprocessing(str(x)))

In [6]:
sample_data = [150,2000,2500]
for i in sample_data:
     print(f"Original Text :\n {df_train.Data[i]} \nCleaned Text: \n {df_train.Text[i]}\n")

Original Text :
 আমারা সবাই একা কিন্তু সবাই মিলে আমরা একা না,,এভাবে একজন একজনের দায়িত্ব নিলে রানারা পাবে একটা সুন্দর জীবন 😍😍  
Cleaned Text: 
 আমারা সবাই একা কিন্তু সবাই মিলে আমরা একা নাএভাবে একজন একজনের দায়িত্ব নিলে রানারা পাবে একটা সুন্দর জীবন  

Original Text :
 ছেলে গুলা জেমন গরদব মেয়ে গুলাও তেমন  
Cleaned Text: 
 ছেলে গুলা জেমন গরদব মেয়ে গুলাও তেমন 

Original Text :
 তুমিই তো শিখিয়েছ অভিজ্ঞতা লাভ ও শিক্ষা পাওয়ার জন্যই আমাদের জন্ম। তুমি বলেছিলে  
Cleaned Text: 
 তুমিই তো শিখিয়েছ অভিজ্ঞতা লাভ ও শিক্ষা পাওয়ার জন্যই আমাদের জন্ম তুমি বলেছিলে 



In [17]:
TEXT_COL = "Text"
LABEL_COLS = ["Love", "Joy", "Surprise", "Anger", "Sadness", "Fear"]
NUM_LABELS = len(LABEL_COLS)

def load_single_file_with_split(path, split_col="split"):
    """Use this instead if you have ONE csv with a column marking train/val/test,
    e.g. df['split'] containing the strings 'train' / 'val' / 'test'."""
    df = pd.read_csv(path)
    df = _clean(df)
    train_df = df[df[split_col].str.lower() == "train"].reset_index(drop=True)
    val_df = df[df[split_col].str.lower() == "val"].reset_index(drop=True)
    test_df = df[df[split_col].str.lower() == "test"].reset_index(drop=True)
    return train_df, val_df, test_df
 
 
def _clean(df):
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    for col in LABEL_COLS:
        df[col] = df[col].astype(int)
    return df
 
 
def get_texts_labels(df):
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COLS].values.astype("float32")
    return texts, labels

# 4. Model

## 4.1 Training XGBoost

In [26]:
X_train_text, y_train = get_texts_labels(df_train)
X_test_text, y_test = get_texts_labels(df_test)
X_val_text, y_val = get_texts_labels(df_val)

In [39]:
vectorizer = TfidfVectorizer(
    max_features = 30000,
    ngram_range = (1, 2)
)

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

In [41]:
baseline = XGBClassifier(
    n_estimators = 300,
    max_depth = 6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree = 0.8,
    eval_metric = "logloss",
    n_jobs = -1,
    random_state = 42
)

model = MultiOutputClassifier(baseline)
model.fit(X_train, y_train)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.8, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=6,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=-1,
                                              num_parallel_tree=None, ...))

In [42]:
print("\n=== Validation performance ===")
val_pred = model.predict(X_val)
print(classification_report(y_val, val_pred, target_names=LABEL_COLS, zero_division=0))
print("Macro F1:", f1_score(y_val, val_pred, average="macro", zero_division=0))
print("Hamming loss:", hamming_loss(y_val, val_pred))
 
print("\n=== Test performance ===")
test_pred = model.predict(X_test)
print(classification_report(y_test, test_pred, target_names=LABEL_COLS, zero_division=0))
print("Macro F1:", f1_score(y_test, test_pred, average="macro", zero_division=0))
print("Hamming loss:", hamming_loss(y_test, test_pred))
 
joblib.dump(vectorizer, "xgb_tfidf_vectorizer.joblib")
joblib.dump(model, "xgb_emotion_model.joblib")
print("\nSaved: xgb_tfidf_vectorizer.joblib, xgb_emotion_model.joblib")


=== Validation performance ===
              precision    recall  f1-score   support

        Love       0.56      0.20      0.29       414
         Joy       0.70      0.57      0.63       942
    Surprise       0.50      0.01      0.02        91
       Anger       0.56      0.14      0.23       388
     Sadness       0.71      0.33      0.45       505
        Fear       0.50      0.03      0.06        32

   micro avg       0.67      0.35      0.47      2372
   macro avg       0.59      0.21      0.28      2372
weighted avg       0.65      0.35      0.44      2372
 samples avg       0.37      0.36      0.36      2372

Macro F1: 0.28058908986646175
Hamming loss: 0.15762905064321772

=== Test performance ===
              precision    recall  f1-score   support

        Love       0.45      0.12      0.19       388
         Joy       0.60      0.50      0.55       856
    Surprise       0.20      0.01      0.01       147
       Anger       0.47      0.11      0.18       548
     Sadne

## 4.2 BiLSTMs